In [37]:
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    fbeta_score
)
from sklearn.model_selection import ParameterGrid

In [38]:
df = pd.read_csv("../../../data/processed/feature_dataset.csv")

In [39]:
# One hot encoding for age and recording location
df = pd.get_dummies(df, columns=["Age"], drop_first=True)

df = pd.get_dummies(df, columns=["recording_location"], drop_first=True)

df = pd.get_dummies(df, columns=["Murmur"], drop_first=True)

df["Sex"] = df["Sex"].map({
    "Female": 0,
    "Male": 1
})

df["Outcome"] = df["Outcome"].map({
    "Normal": 0,
    "Abnormal": 1
})

# print(df.columns.tolist())

In [40]:
# -----------------------------
# Split using your predefined split
# -----------------------------
train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

In [41]:
# -----------------------------
# Prepare features
# -----------------------------
drop_cols = ["Patient ID", "Outcome", "split", "file", "Campaign", "Additional ID", "Height", "Weight"]

drop_cols = drop_cols + ["rms"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["Outcome"]

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["Outcome"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["Outcome"]

## Hyperparameter Tuning
Maximizing F2 score, while keeping minimum accuracy

In [45]:
# ==================================================
# XGBoost Hyperparameter Grid
# ==================================================
#
# max_depth:
#   Maximum depth of each tree.
#   Larger values -> more complex model -> higher overfitting risk.
#
# learning_rate:
#   How much each tree contributes.
#   Lower values are usually safer but require more trees.
#
# n_estimators:
#   Number of trees.
#   More trees can improve performance but increase runtime.
#
# min_child_weight:
#   Minimum amount of data required in a leaf.
#   Larger values make the model more conservative.
#
# subsample:
#   Fraction of training samples used for each tree.
#   Helps reduce overfitting.
#
# colsample_bytree:
#   Fraction of features used for each tree.
#   Similar to Random Forest feature sampling.
#
# scale_pos_weight:
#   Increases importance of the positive class.
#   Useful when abnormal patients are rarer.
#
# Total combinations:
# 3 × 3 × 2 × 2 × 2 × 2 × 3 = 432
#
# ==================================================

param_grid = {
    "max_depth": [2, 3, 4, 5],

    "learning_rate": [0.01, 0.03, 0.05, 0.1],

    "n_estimators": [100, 300, 500],

    "min_child_weight": [1, 3, 5],

    "subsample": [0.8, 1.0],

    "colsample_bytree": [0.8, 1.0],

    "scale_pos_weight": [1, 2, 3, 5],

    "gamma": [0, 0.1, 0.5]
}

# ==================================================
# Selection criterion
# ==================================================
#
# We optimize F2 score
# (recall is weighted 4x more than precision)
#
# But still require a minimum precision of 0.50
#
# ==================================================

min_precision = 0.65

best_model = None
best_params = None
best_threshold = None

best_f2 = -1
best_precision = 0
best_recall = 0

# ==================================================
# Grid Search
# ==================================================

for params in ParameterGrid(param_grid):

    print("Testing:", params)

    model = XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        tree_method="hist",
        n_jobs=-1
    )

    # Train on training set
    model.fit(
        X_train,
        y_train
    )

    # Predicted probabilities on validation set
    y_prob = model.predict_proba(X_val)[:, 1]

    # ==================================================
    # Threshold search
    # ==================================================

    for threshold in np.arange(0.05, 0.95, 0.01):

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        precision = precision_score(
            y_val,
            y_pred,
            zero_division=0
        )

        # Reject models that are too imprecise
        if precision < min_precision:
            continue

        recall = recall_score(
            y_val,
            y_pred
        )

        f2 = fbeta_score(
            y_val,
            y_pred,
            beta=2,
            zero_division=0
        )

        # Keep best F2 score
        if f2 > best_f2:

            best_f2 = f2
            best_precision = precision
            best_recall = recall

            best_threshold = threshold
            best_model = model
            best_params = params.copy()

            print(
                f"NEW BEST | "
                f"F2={f2:.3f} | "
                f"P={precision:.3f} | "
                f"R={recall:.3f} | "
                f"T={threshold:.2f}"
            )

# ==================================================
# Final Results
# ==================================================

print("\n==========================")
print("BEST XGBOOST MODEL")
print("==========================")

print("\nParameters:")
print(best_params)

print(f"\nThreshold: {best_threshold:.2f}")

print("\nValidation Metrics:")
print(f"F2 Score : {best_f2:.3f}")
print(f"Precision: {best_precision:.3f}")
print(f"Recall   : {best_recall:.3f}")

Testing: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 1, 'subsample': 0.8}
NEW BEST | F2=0.641 | P=0.710 | R=0.626 | T=0.45
Testing: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 1, 'subsample': 1.0}
Testing: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 2, 'subsample': 0.8}
Testing: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 2, 'subsample': 1.0}
Testing: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 3, 'subsample': 0.8}
Testing: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child

```py
==========================
BEST XGBOOST MODEL
==========================

Parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 3, 'subsample': 0.8}

Threshold: 0.61


Parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 1, 'subsample': 0.8}

Threshold: 0.42

Validation Metrics:
F2 Score : 0.763
Precision: 0.659
Recall   : 0.795


==========================
BEST XGBOOST MODEL
==========================

Parameters:
{'colsample_bytree': 1.0, 'gamma': 0.5, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 1, 'subsample': 0.8}

Threshold: 0.42

Validation Metrics:
F2 Score : 0.767
Precision: 0.660
Recall   : 0.799

```

In [51]:
X_trainval = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True
)

y_trainval = pd.concat(
    [y_train, y_val],
    axis=0,
    ignore_index=True
)

In [ ]:
xgb_model = XGBClassifier(
    # **best_params,
    colsample_bytree=1.0,
    gamma=0.5,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=1,
    n_estimators=100,
    scale_pos_weight=1,
    subsample=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

best_threshold = 0.42

y_prob_test = xgb_model.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= best_threshold).astype(int)

In [53]:
# EVALUATION
# ONLY RUN ONCE

print("Precision:", precision_score(y_test, y_pred_test))
print("Recall:", recall_score(y_test, y_pred_test))
print("F2:", fbeta_score(y_test, y_pred_test, beta=2))
print("ROC AUC:", roc_auc_score(y_test, y_prob_test))

Precision: 0.604982206405694
Recall: 0.7172995780590717
F2: 0.6916192026037429
ROC AUC: 0.7135608609731346


In [ ]:
import pandas as pd

importance = (
    pd.Series(
        xgb_model.feature_importances_,
        index=X_train.columns
    )
    .sort_values(ascending=False)
)

print(importance.head(20))